### Neo4j notebook
In this notebook we will develop a path-finding algorithm that uses a neo4j database on the transportation network of London as a data source.

## Importing libraries
First of all we need to import the various libraries we are going to use in this notebook:
- ***os***: it helps to install the needed libraries if absent
- ***random***: it will be used to shuffle some data
- ***math***: it will be used to compute the distance in meters between two pairs of coordinates
- ***neo4j***: allows to connect to a neo4j server and to perform queries
- ***neo4jupyter***: allows to visualize queries as graphs in python
- ***py2neo***: allows to work with neo4jupyter

In [1]:
import os
import random
import math

try:
    import neo4j as nj
except:
    os.system("pip install neo4j")
    
try:
    import py2neo
except:
    os.system("pip install py2neo")

try:
    import neo4jupyter
except:
    os.system("pip install neo4jupyter")
    
neo4jupyter.init_notebook_mode()

<IPython.core.display.Javascript object>

We can now connect to the DataBase:

In [2]:
driver = nj.GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password"))
graph = py2neo.Graph("bolt://localhost:7687", auth=("neo4j", "password"))

Let's check if we connected properly:

In [6]:
record = graph.run("MATCH (n) RETURN n limit 100")
neo4jupyter.draw(graph= graph, limit=10, options={"Station": "name", "BusStop": "name"})

Now that we are properly connected we can start working on the aim of the notebook: developing a path finding algorithm from one stop to another.

## Declaring functions
We will now declare some useful functions that will be useful later on in the path finding algorithm.

These functions work with coordinates:
- *haversine_distance*: computes the distance in meters of two points given their coordinates
- *coordinate_difference_for_distance*: computes the maximum coordinates someone can go before walking more than a given amount of meters, starting from given coordinates (only latitude is necessary)

In [3]:
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    R = 6371000  # meters

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    # hevrsine formula
    a = math.sin(delta_lat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(delta_lon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Calculate distance
    distance = R * c

    return distance


def coordinate_difference_for_distance(lat, distance):
    R = 6371000  # meters

    # Differenza in latitudine (indipendente dalla latitudine)
    delta_lat = (distance / R) * (180 / math.pi)

    # Differenza in longitudine (dipendente dalla latitudine)
    delta_lon = (distance / (R * math.cos(math.radians(lat)))) * (180 / math.pi)

    return delta_lat, delta_lon

- *retrieve_stop*: returns the stop data extracted from the database given the stop name
- *find_directly_connected_stop*: returns all the stops directly connected to the given stop

In [94]:
def retrieve_stop(station_name):
    query = "MATCH (n) WHERE n.name = $station_name RETURN n"
    
    result = driver.session().run(query, station_name=station_name)
        
    stops = [record["n"] for record in result]
        
    
    return [{"id": node.element_id, **node._properties} for node in stops]

def find_directly_connected_stop(from_station):
    query = "MATCH (n {name: $stat_name})-[*]-(reachable) RETURN DISTINCT reachable"
    
    result = driver.session().run(query, stat_name=from_station)
    
    conn_stops = [node["n"] for node in result]
    conn_stops = [{"id": node.element_id, **node._properties} for node in conn_stops]
    
    return conn_stops

- *extract_paths*: extract the data from a path handed out from the DB and organizes them in: path, stations visited and lines used
- *print_path*: prints the final path found thanks to the algorithm 

In [95]:
def extract_paths (cursor):
    path = []
    stops = []
    lines = []
    for result in cursor:
        this_path = result['p']
        stops = [{'name': node.element_id, **node._properties} for node in this_path.nodes]
        lines = [
                    {
                        "id": rel.element_id,
                        "start": rel.start_node.element_id,
                        "end": rel.end_node.element_id,
                        "type": rel.type,
                        **rel._properties
                    }
                    for rel in this_path.relationships
                ]
        path.append({"stations": stops, "lines": lines})
    
    return path, stops, lines

def print_path(end_stop, stations, lines, changes):
    names = [stop['name'] for stop in stations]
    ln = [line['route'] for line in lines]
    
    for key in changes.keys():
        value = changes[key]
        path = value[1]
        changes_names = [key, [stop['name'] for stop in path[0]]]
        changes_line = [line['route'] for line in path[1]]
    
    for i in range((len(changes_names)//2)):
        print("Starting from " + changes_names[i])
        if len(changes_names[i+1])==0:
            print(" you walk to ")
            if len(changes_names)>i+2:
                print(changes_names[i+2])
            else:
                print(names[0])
        else:
            print(" You visit the following stops: ")
            print(changes_names[i+1])
            print()
            print(" Using the following lines: ")
            print(changes_line)
            print()
            print(" Arriving in:")
            if len(changes_names)>i+2:
                print(changes_names[i+2])
            else:
                print(names[0])
    
    print()
    print()
    print("Starting from " + names[0])
    print(" you visit the following stops: ")
    print(names)
    print()
    print(" Using the following lines: ")
    print(ln)
    print()
    print(" Arriving in:")
    print(end_stop)

Lastly we print some random stops, so that anyone can use and try by themselves the following algorithm. If a stop name is in uppercase then it represents a bus stop, on the other hand it represents an underground station

In [88]:
query = "MATCH (n) WITH n, rand() AS random ORDER BY random LIMIT 100 OPTIONAL MATCH (n)-[r]->(m) RETURN n.name"

result = driver.session().run(query)

for record in result:
    print(record['n.name'])

DORKING SPORTS CENTRE
POTTERS CLOSE
HATTON GREEN
MITCHAM ROAD / TOOTING BROADWAY STN <>
CLAREMONT ROAD / SURBITON STATION #
NORWICH ROAD
CARLISLE ROAD
SIPSON ROAD
BLENHEIM ROAD
CRANBROOK PRIMARY SCHOOL
YE OLD GEORGE INN
IMPERIAL COLLEGE/QUEENS GATE TERRACE
HARLINGTON ROAD WEST
HARLINGTON ROAD WEST
WIBBANDUNE SPORTS CLUB
THE RAVENSBURY ARMS
WEST BROMPTON STATION <>
CEDARS ROAD
CEDARS ROAD
NEW CROSS STATION #
NEW CROSS STATION #
NEW CROSS STATION #
SYDENHAM RISE
MAPLEDOWN SCHOOL
THICKET CRESCENT
NORTHOLT HIGH SCHOOL
REVESBY ROAD
PLUMSTEAD COMMON / THE SHIP
CHURCH ROAD
CHURCH ROAD
STANSTEAD ROAD
WALSINGHAM ROAD
CLAYPONDS AVENUE
GOLF LINKS ESTATE
ST PAUL'S WAY TRUST SCHOOL
LOWER DOWNS ROAD
WOODHAYES ROAD
WOODHAYES ROAD
TOOTING BEC STATION <>
TOOTING BEC STATION <>
SHIRLAND ROAD
HIGH STREET / SOUTHALL TOWN HALL
HIGH STREET / SOUTHALL TOWN HALL
HIGH STREET / SOUTHALL TOWN HALL
WEARSIDE ROAD
WEARSIDE ROAD
NORTH FINCHLEY HIGH ROAD
DU CANE COURT
THE AVENUE
ORCHARDSON STREET
CAMDEN HIGH STREET
C

## Path finding algorithm

The following algorithm aims to find a path from a bus stop or an underground station to another. The challenging part of this algorithm consists in considering those paths in which the person needs to walk from one stop to another, since not every stop is directly connected to any other. Since it is supposed to be a realistic algorithm, we consider paths in which someone does not have to walk more than 3 times.

In [99]:
max_walking_distance = 250 #meters
max_walking_times = 3

def find_a_path(starting_node, end_node):
    #initialization
    stops_queue = []
    visited = set()
    start_stop = retrieve_stop(starting_node)[0]
    start_stop['changes'] = {}
    stops_queue.append(start_stop)
    
    while len(stops_queue) != 0:
        stop = stops_queue.pop(0)
        if len(stop['changes']) >= max_walking_times:
            #we do not consider paths in which you have to make more than max_walking_times changes by foot
            continue
            
        if stop['id'] not in visited:
            visited.add(stop['id'])
        else:
            #we already considered this station
            continue
        
        query = "MATCH p = shortestPath((startNode)-[*]-(endNode)) WHERE startNode.name = $starting_node AND endNode.name= $ending_node RETURN p"
        result = driver.session().run(query, starting_node = stop['name'], ending_node = end_node)
        path, stops, lines = extract_paths(result)
        
        if len(stops) != 0:
            #we found a path
            break
            
        #find directly connected stops and add them to the queue
        directly_connected_stations = find_directly_connected_stop(stop)
        random.shuffle(directly_connected_stations)
        for node in directly_connected_stations:
            node['changes'] = stop['changes']
            stops_queue.append(node)
        
        #calculate max walking distance
        delta_lat, delta_lon = coordinate_difference_for_distance(stop['latitude'], max_walking_distance)
        min_lat = stop['latitude'] - delta_lat
        max_lat = stop['latitude'] + delta_lat
        min_long = stop['longitude'] - delta_lon
        max_long = stop['longitude'] + delta_lon
        
        #look for neighbouring station that can be walked to
        query = "MATCH (n) WHERE elementId(n) <> $stop_id AND n.name <> $stop_name AND n.latitude<$max_lat AND n.latitude>$min_lat AND n.longitude<$max_long AND n.longitude>$min_long RETURN DISTINCT n"
        result = driver.session().run(query, stop_name = stop['name'], stop_id = stop['id'], max_lat = max_lat, max_long = max_long, min_lat = min_lat, min_long = min_long)
        
        walking_stops = [node["n"] for node in result]
        walking_stops = [{"id": node.element_id, **node._properties} for node in walking_stops]
        random.shuffle(walking_stops)
        #adding the current station to the neighbouring walking stations changes list (since we walked from station to node by foot)
        for node in walking_stops:
            node['changes'] = stop['changes']
            
            if len(node['changes']) != 0:
                last_changes_inserted_key = list(node['changes'].keys())[-1]
                last_value = node['changes'][last_changes_inserted_key]
                last_name = last_value[0]
            else:
                last_name = stop['name']
            
            #find the path leading to the station in which you have to walk to another stop, starting from the current station or from the previous stop we walked to    
            if last_name != node['name']:          
                query = "MATCH p = shortestPath((startNode)-[*]-(endNode)) WHERE startNode.name = $starting_node AND endNode.name= $ending_node RETURN p"
                changes_result = driver.session().run(query, starting_node = last_name, ending_node = node['name'])
                n_path, n_stations, n_lines = extract_paths(changes_result)
                
                node['changes'].update({stop['name']: (node['name'], [n_stations, n_lines])})
        
        stops_queue.extend(walking_stops)
        
    return path, stops, lines, stop['changes']


We can now try our algorithm.

In [105]:
starting_stop = "Woolwich" #here you can enter whichever station you want
ending_stop = "WESTMINSTER STN <> / PARLIAMENT SQUARE" #here you can enter whichever station you want
pathf, stationsf, linesf, change = find_a_path(starting_stop, ending_stop)

print_path(ending_stop, stationsf, linesf, change)

Starting from Woolwich
 You visit the following stops: 
['PLUMSTEAD ROAD / BURRAGE ROAD', 'BURRAGE ROAD', "BERESFORD SQ / WOOL'CH ARSENAL STN [DLR]", 'WOOLWICH ARSENAL STATION # [DLR]', 'PLUMSTEAD ROAD / WOOLWICH PUBLIC MARKET']

 Using the following lines: 
['54', '51', '161', '244']

 Arriving in:
VINCENT ROAD / WOOLWICH ARSENAL STATION


Starting from VINCENT ROAD / WOOLWICH ARSENAL STATION
 you visit the following stops: 
['VINCENT ROAD / WOOLWICH ARSENAL STATION', 'WOOLWICH ARSENAL STATION # [DLR]', 'CALDERWOOD STREET', 'ARTILLERY PLACE / FRANCES STREET', 'REPOSITORY ROAD / ARTILLERY PLACE', 'ERWOOD ROAD', 'CEMETERY LANE', 'CHARLTON PARK ROAD', 'CHARLTON CHURCH LANE', 'CHARLTON ROAD / VICTORIA WAY', 'WYNDCLIFF ROAD', 'BLACKHEATH / ROYAL STANDARD', 'VANBRUGH PARK / STRATHEDEN ROAD', 'VANBRUGH PARK / BEACONSFIELD ROAD', 'PRINCE CHARLES ROAD / MAZE HILL', 'GREENWICH PARK', 'WAT TYLER ROAD', 'ORCHARD ROAD', 'DARTMOUTH HILL', 'ELIOT HILL', 'LEWISHAM STATION #', 'MOLESWORTH STREET', 'MO

This algorithm is unfortunately far from perfect, there are many improving aspects but satisfies it's goal of finding a path from one stop to another. The aspects in which there's room for improvement are:
- preferring taking the underground instead of doing a long journey using the bus
- taking into consideration how much time it takes to go from one stop to an adjacent one
- changing the order visit of nodes in order to prefer those who are closer to the end stop
- considering more than one path and then confront them